# Módulo 41: Importación y consultas SQL

Autor: Juliho Castillo Colmenares

> Lección complementaria del ejercicio [m41-importacion-y-consultas-sql.ipynb](m41-importacion-y-consultas-sql.ipynb). No es una tarea entregada del programa.

En el Módulo 40 cargamos un CSV pequeño fila por fila. Con archivos de miles de filas y muchas columnas es más práctico dejar que pandas cree la tabla: `DataFrame.to_sql` la genera y la llena en una sola llamada. En esta lección usamos `recursos-humanos.csv`, con unos 15 000 empleados de una empresa, para importar los datos, revisar cómo quedaron guardados y responder preguntas con consultas de agregación.

1. Importar un DataFrame con `to_sql`
2. `to_sql` frente a `CREATE TABLE` + `INSERT`
3. Afinidad de tipos tras la importación
4. Nombres de columnas problemáticos
5. Leer resultados: `fetchone` y `fetchall`
6. Agregados: `AVG` y compañía
7. Filtros con `IN`, `AND` y `OR`
8. Errores frecuentes
9. Ejercicio

In [1]:
import sqlite3

import pandas as pd

datos = pd.read_csv("recursos-humanos.csv")
print(datos.shape)
datos.head()

(14999, 10)


,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,sales,salary
0,0.38,0.53,2,157,3,0,1,0,sales,low
1,0.80,0.86,5,262,6,0,1,0,sales,medium
2,0.11,0.88,7,272,4,0,1,0,sales,medium
3,0.72,0.87,5,223,5,0,1,0,sales,low
4,0.37,0.52,2,159,3,0,1,0,sales,low


Cada fila es un empleado. `left` vale 1 si dejó la empresa, `sales` es en realidad el **departamento** y `salary` es un nivel (`low`, `medium`, `high`), no un monto.

## 1. Importar un DataFrame con `to_sql`

`df.to_sql(nombre, conn, if_exists=..., index=...)` crea la tabla a partir de las columnas y tipos del DataFrame e inserta todas las filas.

| Parámetro | Valores | Efecto |
| --- | --- | --- |
| `if_exists` | `"fail"` (por defecto) | Error si la tabla ya existe |
| | `"replace"` | **Borra** la tabla existente y la crea de nuevo |
| | `"append"` | Agrega las filas a la tabla existente, respetando su estructura |
| `index` | `True` (por defecto) / `False` | Guarda o no el índice del DataFrame como una columna más |
| `dtype` | Diccionario columna → tipo SQL | Fuerza el tipo declarado de algunas columnas |

`to_sql` devuelve el número de filas insertadas.

In [2]:
conn = sqlite3.connect(":memory:")

filas = datos.to_sql("Empleado", conn, if_exists="replace", index=False)
print("Filas insertadas:", filas)

Filas insertadas: 14999


## 2. `to_sql` frente a `CREATE TABLE` + `INSERT`

| Enfoque | Ventajas | Limitaciones |
| --- | --- | --- |
| `to_sql` | Una línea; deduce los tipos del DataFrame | Sin clave primaria, `NOT NULL` ni `CHECK` |
| `CREATE TABLE` + `INSERT` (o `executemany`) | Control total de tipos y restricciones | Más código |
| `CREATE TABLE` + `to_sql(if_exists="append")` | Las restricciones de la tabla y la comodidad de pandas | Las columnas deben coincidir |

La tabla que crea `to_sql` es la más simple posible:

In [3]:
print(conn.execute("SELECT sql FROM sqlite_master WHERE name = 'Empleado'").fetchone()[0])

CREATE TABLE "Empleado" (
"satisfaction_level" REAL,
  "last_evaluation" REAL,
  "number_project" INTEGER,
  "average_montly_hours" INTEGER,
  "time_spend_company" INTEGER,
  "Work_accident" INTEGER,
  "left" INTEGER,
  "promotion_last_5years" INTEGER,
  "sales" TEXT,
  "salary" TEXT
)


Si se necesita una tabla con restricciones, se crea primero y se llena con `if_exists="append"`. Con `"replace"`, en cambio, `to_sql` **borra** la tabla creada a mano y la sustituye por la suya, de modo que el `CREATE TABLE` previo no sirve de nada:

In [4]:
muestra = datos.head(3)[["satisfaction_level", "salary"]]

conn.execute("""
CREATE TABLE Encuesta (
    satisfaction_level REAL NOT NULL CHECK (satisfaction_level BETWEEN 0 AND 1),
    salary             TEXT CHECK (salary IN ('low', 'medium', 'high'))
)""")
muestra.to_sql("Encuesta", conn, if_exists="append", index=False)
print("Con append: ", conn.execute("SELECT sql FROM sqlite_master WHERE name = 'Encuesta'").fetchone()[0][:60], "...")

muestra.to_sql("Encuesta", conn, if_exists="replace", index=False)
print("Con replace:", conn.execute("SELECT sql FROM sqlite_master WHERE name = 'Encuesta'").fetchone()[0])

Con append:  CREATE TABLE Encuesta (
    satisfaction_level REAL NOT NULL ...
Con replace: CREATE TABLE "Encuesta" (
"satisfaction_level" REAL,
  "salary" TEXT
)


Tras el `replace`, los `CHECK` desaparecieron. Con `append`, la tabla también rechaza filas inválidas:

In [5]:
conn.execute("DROP TABLE Encuesta")
conn.execute("""
CREATE TABLE Encuesta (
    satisfaction_level REAL NOT NULL CHECK (satisfaction_level BETWEEN 0 AND 1),
    salary             TEXT CHECK (salary IN ('low', 'medium', 'high'))
)""")
invalida = pd.DataFrame({"satisfaction_level": [1.7], "salary": ["low"]})
try:
    invalida.to_sql("Encuesta", conn, if_exists="append", index=False)
except Exception as error:
    print("Rechazada:", type(error).__name__, "-", error)

Rechazada: IntegrityError - CHECK constraint failed: satisfaction_level BETWEEN 0 AND 1


## 3. Afinidad de tipos tras la importación

`to_sql` traduce los tipos de pandas a tipos SQL, y SQLite les asigna una afinidad:

| dtype de pandas | Tipo declarado | Afinidad |
| --- | --- | --- |
| `int64` | `INTEGER` | INTEGER |
| `float64` | `REAL` | REAL |
| `object` (texto) | `TEXT` | TEXT |
| `bool` | `INTEGER` | INTEGER (0/1) |

Conviene revisar los tipos con `typeof` justo después de importar: si una columna numérica llega como texto (por ejemplo, porque el CSV tenía un símbolo de moneda), las comparaciones y los promedios fallan en silencio.

In [6]:
pd.read_sql_query("""
SELECT typeof(satisfaction_level) AS satisfaccion,
       typeof(number_project)     AS proyectos,
       typeof(salary)             AS salario
FROM Empleado
LIMIT 1
""", conn)

,satisfaccion,proyectos,salario
0,real,integer,text


## 4. Nombres de columnas problemáticos

Los nombres de este archivo traen varios problemas típicos:

| Columna | Problema | Nombre propuesto |
| --- | --- | --- |
| `average_montly_hours` | Error ortográfico ("montly") | `average_monthly_hours` |
| `Work_accident` | Mayúscula inconsistente con el resto | `work_accident` |
| `sales` | No describe el contenido (es el departamento) | `department` |
| `left` | Palabra reservada de SQL (`LEFT JOIN`); en T-SQL exige `[left]` | `has_left` |

Lo más limpio es renombrar en pandas **antes** de importar. Si la tabla ya existe, se usa `ALTER TABLE ... RENAME COLUMN`.

In [7]:
datos = datos.rename(columns={
    "average_montly_hours": "average_monthly_hours",
    "Work_accident": "work_accident",
    "sales": "department",
    "left": "has_left",
})
datos.to_sql("Empleado", conn, if_exists="replace", index=False)

print([fila[1] for fila in conn.execute("PRAGMA table_info(Empleado)")])

['satisfaction_level', 'last_evaluation', 'number_project', 'average_monthly_hours', 'time_spend_company', 'work_accident', 'has_left', 'promotion_last_5years', 'department', 'salary']


Si no se renombra, una palabra reservada se puede usar entre comillas dobles (`"left"`) en SQLite o entre corchetes (`[left]`) en SQL Server. SQLite acepta `left` sin comillas en muchas consultas, pero no conviene depender de esa tolerancia.

## 5. Leer resultados: `fetchone` y `fetchall`

Una consulta de agregado sin `GROUP BY` devuelve una sola fila. `fetchone()` la entrega como tupla, y `[0]` extrae el primer valor. `fetchall()` devuelve todas las filas en una lista.

In [8]:
cur = conn.cursor()

cur.execute("SELECT COUNT(*), AVG(satisfaction_level) FROM Empleado")
total, satisfaccion = cur.fetchone()
print(f"{total} empleados, satisfacción promedio {satisfaccion:.3f}")

cur.execute("SELECT DISTINCT salary FROM Empleado ORDER BY salary")
print(cur.fetchall())

14999 empleados, satisfacción promedio 0.613
[('high',), ('low',), ('medium',)]


## 6. Agregados: `AVG` y compañía

`AVG` sobre una columna que vale 0 o 1 da la **proporción** de unos. Así, `AVG(has_left)` es la tasa de rotación:

$$\text{tasa de rotación} = \frac{\text{empleados que se fueron}}{\text{total de empleados}}$$

Con `GROUP BY` se compara entre grupos, y `ROUND` deja el resultado legible.

In [9]:
pd.read_sql_query("""
SELECT has_left,
       COUNT(*)                            AS empleados,
       ROUND(AVG(satisfaction_level), 3)   AS satisfaccion,
       ROUND(AVG(average_monthly_hours), 1) AS horas_mes
FROM Empleado
GROUP BY has_left
""", conn)

,has_left,empleados,satisfaccion,horas_mes
0,0,11428,0.667,199.1
1,1,3571,0.440,207.4


In [10]:
pd.read_sql_query("""
SELECT salary,
       COUNT(*)                        AS empleados,
       ROUND(100.0 * AVG(has_left), 1) AS pct_rotacion
FROM Empleado
GROUP BY salary
ORDER BY pct_rotacion DESC
""", conn)

,salary,empleados,pct_rotacion
0,low,7316,29.7
1,medium,6446,20.4
2,high,1237,6.6


La rotación disminuye a medida que sube el nivel salarial: el grupo `low` es el que más empleados pierde y `high` el que menos.

## 7. Filtros con `IN`, `AND` y `OR`

`IN` resume varias igualdades sobre la misma columna. Cuando se mezclan `AND` y `OR`, **`AND` se evalúa primero**, igual que la multiplicación antes que la suma. Por eso conviene usar paréntesis siempre que aparezcan los dos.

In [11]:
cur.execute("""
SELECT COUNT(*), ROUND(AVG(average_monthly_hours), 2)
FROM Empleado
WHERE salary IN ('low', 'medium') AND has_left = 1
""")
print("Bajo o medio que se fueron (cantidad, horas):", cur.fetchone())

cur.execute("""
SELECT COUNT(*)
FROM Empleado
WHERE (salary = 'low' OR salary = 'medium') AND has_left = 1
""")
print("La misma condición con OR y paréntesis:", cur.fetchone()[0])

Bajo o medio que se fueron (cantidad, horas): (3489, 207.76)
La misma condición con OR y paréntesis: 3489


## 8. Errores frecuentes

**Mezclar `AND` y `OR` sin paréntesis.** La condición sin paréntesis se lee como `salary = 'low' OR (salary = 'medium' AND has_left = 1)`: incluye a **todos** los de salario bajo, se hayan ido o no.

In [12]:
sin_parentesis = conn.execute(
    "SELECT COUNT(*) FROM Empleado WHERE salary = 'low' OR salary = 'medium' AND has_left = 1"
).fetchone()[0]
con_parentesis = conn.execute(
    "SELECT COUNT(*) FROM Empleado WHERE (salary = 'low' OR salary = 'medium') AND has_left = 1"
).fetchone()[0]
print("Sin paréntesis:", sin_parentesis, "| con paréntesis:", con_parentesis)

Sin paréntesis: 8633 | con paréntesis: 3489


**Olvidar `index=False`.** Con el valor por defecto, `to_sql` agrega una columna `index` con el índice del DataFrame, que no forma parte de los datos:

In [13]:
datos.head(2).to_sql("ConIndice", conn, if_exists="replace")
print([fila[1] for fila in conn.execute("PRAGMA table_info(ConIndice)")][:3])

['index', 'satisfaction_level', 'last_evaluation']


**Escribir un `CREATE TABLE` y luego usar `if_exists="replace"`.** Como se vio en la sección 2, `replace` descarta la tabla creada a mano junto con sus restricciones. Se usa `"append"` si la tabla se definió antes, o solo `to_sql` si no hacen falta restricciones.

**Usar nombres con errores o palabras reservadas.** Cada consulta posterior arrastra el problema (`average_montly_hours`, `[left]`). Renombrar una vez, al importar, evita errores en todas las siguientes.

## 9. Ejercicio

Con la tabla `Empleado` ya renombrada:

1. Cuenta cuántos empleados tuvieron un accidente laboral (`work_accident = 1`) y calcula la tasa de rotación de ese grupo y la del resto.
2. Obtén la satisfacción promedio de los empleados con 6 o más proyectos que trabajan más de 250 horas al mes.
3. Lista los departamentos con su número de empleados y su tasa de rotación, solo para los departamentos `sales`, `technical` y `support`, ordenados por rotación.
4. Cuenta los empleados promovidos en los últimos 5 años que tienen salario `high` o que se quedaron en la empresa. Escribe la condición con paréntesis explícitos.

### Solución

In [14]:
pd.read_sql_query("""
SELECT work_accident,
       COUNT(*)                        AS empleados,
       ROUND(100.0 * AVG(has_left), 1) AS pct_rotacion
FROM Empleado
GROUP BY work_accident
""", conn)

,work_accident,empleados,pct_rotacion
0,0,12830,26.5
1,1,2169,7.8


In [15]:
cur.execute("""
SELECT COUNT(*), ROUND(AVG(satisfaction_level), 3)
FROM Empleado
WHERE number_project >= 6 AND average_monthly_hours > 250
""")
cantidad, satisfaccion = cur.fetchone()
print(f"{cantidad} empleados con satisfacción promedio {satisfaccion}")

856 empleados con satisfacción promedio 0.146


In [16]:
display(pd.read_sql_query("""
SELECT department,
       COUNT(*)                        AS empleados,
       ROUND(100.0 * AVG(has_left), 1) AS pct_rotacion
FROM Empleado
WHERE department IN ('sales', 'technical', 'support')
GROUP BY department
ORDER BY pct_rotacion DESC
""", conn))

cur.execute("""
SELECT COUNT(*)
FROM Empleado
WHERE promotion_last_5years = 1 AND (salary = 'high' OR has_left = 0)
""")
print("Promovidos con salario alto o que se quedaron:", cur.fetchone()[0])

,department,empleados,pct_rotacion
0,technical,2720,25.6
1,support,2229,24.9
2,sales,4140,24.5


Promovidos con salario alto o que se quedaron: 300


In [17]:
conn.close()

## Resumen

- `to_sql` crea y llena una tabla desde un DataFrame; `index=False` evita la columna del índice.
- `if_exists="replace"` borra la tabla existente, restricciones incluidas; `"append"` respeta una tabla definida con `CREATE TABLE`.
- Tras importar conviene revisar el esquema (`sqlite_master`, `PRAGMA table_info`) y los tipos (`typeof`).
- Los nombres con errores o palabras reservadas se corrigen al importar, con `rename` en pandas o `ALTER TABLE ... RENAME COLUMN`.
- `AVG` de una columna 0/1 da una proporción; `AND` se evalúa antes que `OR`, así que los paréntesis no son opcionales.

En el **Módulo 42** se profundiza en `ORDER BY` con varias claves, `GROUP BY` con varias columnas y `HAVING`, llevando los resultados a pandas con `read_sql_query`.